In [1]:
from pathlib import Path
import sys

# Find the repository root from the notebook's current directory.
project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (
            path
            / "scripts"
            / "graph_building"
            / "wasserstein_distance_graph2.py"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError(
        f"Could not locate the repository root from {Path.cwd()}"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: /home/lucagalli/Projects/parcellating_dbm


In [2]:
"""
For one subject, load parcel log-Jacobian distributions and compute the
parcel-to-parcel similarity matrix in memory.

Construct MST-constrained graphs at 8%, 10%, and 12% density, compute global
and nodal graph metrics immediately, save only the resulting metric vectors
and summaries, and discard all adjacency data.
"""

# Build a pilot for a single subject only for testing

import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
# Import the graph building functions
from scripts.graph_building.wasserstein_distance_graph2 import (
    SIM_FORMULA_INV1PW,
    _compute_block,
    load_subject_parcels,
)


In [3]:
pilot_subject = "sub-0001"
pilot_nodes = 2_000
block_size = 250

subject_dir = (
    project_root / "outputs" / "jacobian_parcel_vectors"
    / pilot_subject
)

In [4]:
parcel_ids, quantile_matrix, parcel_stats = load_subject_parcels(
    subject_dir
)

total_parcels = len(parcel_ids)
n_nodes = min(pilot_nodes, len(parcel_ids))

parcel_ids = parcel_ids[:n_nodes]
quantile_matrix = quantile_matrix[:n_nodes, :]
parcel_stats = {name: values[:n_nodes] for name, values in parcel_stats.items()}

print("Available parcels:", total_parcels)
print("Pilot quantile matrix:", quantile_matrix.shape)


Available parcels: 83442
Pilot quantile matrix: (2000, 15)


In [5]:
similarity_matrix = np.empty(
    (n_nodes, n_nodes),
    dtype=np.float32,
)

# Loop over the blocks of the similarity matrix
for start in range(0, n_nodes, block_size):

    # Compute the end of the current block
    end = min(start + block_size, n_nodes)

    # Compute the similarity block for the current block
    _, _, similarity_block = _compute_block(
        start,
        end,
        quantile_matrix,
        SIM_FORMULA_INV1PW,  # similarity = 1 / (1 + Wasserstein)
    )

    # Store the similarity block in the similarity matrix
    similarity_matrix[start:end, :] = similarity_block

print("Similarity matrix:", similarity_matrix.shape)
print("Similarity range:", similarity_matrix.min(), similarity_matrix.max())
print(
    "Symmetric:",
    np.allclose(similarity_matrix, similarity_matrix.T),
)

Similarity matrix: (2000, 2000)
Similarity range: 0.29448974 1.0
Symmetric: True


In [6]:
cost_matrix = 1.0 - similarity_matrix

# Self-connections must not be part of the MST.
np.fill_diagonal(cost_matrix, 0.0)

mst = minimum_spanning_tree(cost_matrix)

# scipy returns a directed sparse representation of the undirected MST.
mst = mst + mst.T

print("MST shape:", mst.shape)
print("MST undirected entries:", mst.nnz)
print("Expected entries:", 2 * (n_nodes - 1))

MST shape: (2000, 2000)
MST undirected entries: 3998
Expected entries: 3998
